pip install ONE-api
pip install ibllib

In [2]:
srun --partition=gpu --gres=gpu:1 --nodes=1 --ntasks=1 --time=00:10:00 --pty bash  #gpu 파티션에서 GPU 1개를 최대 10분간 할당받음

SyntaxError: cannot assign to expression (2224679219.py, line 1)

In [ ]:
hostname
nvidia-smi
"$LFADS_PYTHON" -c "import torch; print('PyTorch:', torch.__version__); print('CUDA build:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available())"
#homename으로 실제 배정된 node 확인, nvidia-smi로 그 node의 GPU 상태 확인, CUDA avaliable: Ture가 되면 CUDA 사용 가능

In [ ]:
conda create -n lfads-torch python=3.9 -y
conda activate lfads-torch

In [ ]:
python -m pip install "numpy<2"

python -m pip install torch==1.13.1+cu117 \
  --extra-index-url https://download.pytorch.org/whl/cu117
#CUDA 지원하는 PyTorch 설치

In [ ]:
python -c "import torch; print('Torch:', torch.__version__); print('CUDA:', torch.version.cuda); print('GPU available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available')"
#GPU를 배정받은 node에서 lfads-torch 환경을 활성화 후 확인

In [ ]:
from one.api import ONE

In [ ]:
ONE.setup(base_url='https://openalyx.internationalbrainlab.org', silent=True)

IBLParams(CLIENT_MAP={'openalyx.internationalbrainlab.org': '/home/changhyun/Downloads/ONE/openalyx.internationalbrainlab.org'}, DEFAULT='openalyx.internationalbrainlab.org')

In [ ]:
one = ONE(password='international')

Connected to https://openalyx.internationalbrainlab.org as user "intbrainlab"


In [ ]:
import requests

r = requests.get("https://openalyx.internationalbrainlab.org", timeout=10)
print(r.status_code)

200


In [ ]:
insertions = one.search_insertions(
    atlas_acronym='MOp',
    datasets='spikes.times.npy',
    project='brainwide'
)

print(len(insertions))
print(insertions[:5])

37
[UUID('0ffa052a-fe3f-46c2-999d-43bd56b7826e'), UUID('edc82c06-83fd-44be-a41e-52fdeea728b4'), UUID('310c60b6-d68f-4018-a86a-3668ce296837'), UUID('09b62a15-dd8b-4f47-881f-d3f1e5ea5ff4'), UUID('7c94d733-b913-4064-83f2-37422712204c')]


In [ ]:
from brainbox.io.one import SpikeSortingLoader

pid = str(insertions[0])

ssl = SpikeSortingLoader(pid=pid, one=one)

spikes, clusters, channels = ssl.load_spike_sorting(
    revision='2024-05-06'
)

clusters = ssl.merge_clusters(spikes, clusters, channels)

In [ ]:
import numpy as np

print(clusters.keys())
print(np.unique(clusters['acronym']))

#M1 neuron 갯수

dict_keys(['channels', 'depths', 'uuids', 'cluster_id', 'amp_max', 'amp_min', 'amp_median', 'amp_std_dB', 'contamination', 'contamination_alt', 'drift', 'missed_spikes_est', 'noise_cutoff', 'presence_ratio', 'presence_ratio_std', 'slidingRP_viol', 'spike_count', 'slidingRP_viol_forced', 'max_confidence', 'min_contamination', 'n_spikes_below2', 'firing_rate', 'label', 'bitwise_fail', 'ks2_label', 'x', 'y', 'z', 'acronym', 'atlas_id', 'axial_um', 'lateral_um', 'labels', 'rawInd'])
['AD' 'AHN' 'AMd' 'AMv' 'AV' 'HY' 'MOp6a' 'MOp6b' 'PR' 'PVHd' 'PeF' 'VL'
 'VM' 'VMH' 'ZI' 'ccb' 'chpl' 'cing' 'fi']


In [ ]:
acr = np.asarray(clusters['acronym']).astype(str)

m1_mask = np.char.startswith(acr, 'MOp')
good_mask = clusters['label'] == 1

m1_good_mask = m1_mask & good_mask
m1_cids = np.where(m1_good_mask)[0]

print("All clusters:", len(acr))
print("M1 clusters:", np.sum(m1_mask))
print("Good M1 clusters:", len(m1_cids))
print("M1 regions:", np.unique(acr[m1_mask]))
print("Selected cluster IDs:", m1_cids[:20])

All clusters: 603
M1 clusters: 31
Good M1 clusters: 5
M1 regions: ['MOp6a' 'MOp6b']
Selected cluster IDs: [487 493 498 500 503]


In [ ]:
eid, pname = one.pid2eid(pid)

print("Session ID:", eid)
print("Probe:", pname)

trials = one.load_object(eid, 'trials')

print(trials.keys())
print("Number of trials:", len(trials['choice']))

#이 probe가 속한 session의 trial 정보를 가져오기

Session ID: 7f150b7c-c261-46e6-9edb-cc391c9d9f03
Probe: probe00


/home/changhyun/LFADS_pra/.venv/lib/python3.10/site-packages/one/util.py:464: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


dict_keys(['goCueTrigger_times', 'intervals_bpod', 'quiescencePeriod', 'stimOffTrigger_times', 'stimOff_times', 'stimOnTrigger_times', 'goCue_times', 'response_times', 'choice', 'stimOn_times', 'contrastLeft', 'contrastRight', 'feedback_times', 'feedbackType', 'rewardVolume', 'probabilityLeft', 'firstMovement_times', 'intervals'])
Number of trials: 656


In [ ]:
trials['firstMovement_times']
#L>기준점
#M1이면 일단 movement onset에 맞춰보는 걸 추천

array([  67.53868499,   71.33068499,   74.13268499,   79.63968499,
         84.52468499,   88.42568499,   92.24668499,  101.19268499,
        104.26768499,  107.06868499,  110.29568499,  116.25268499,
        120.96468499,  126.18368499,  131.58768499,  135.38168499,
        139.49668499,  143.05068499,  146.39368499,  149.97168499,
        156.71068499,  161.20868499,  164.53168499,  169.85268499,
        174.82768499,  179.51268499,  182.23468499,  187.00468499,
        189.83968499,  194.91868499,  200.54668499,  207.04668499,
        211.79168499,  218.65268499,  223.55468499,  226.92768499,
        231.52068499,  238.08368499,  242.32868499,  247.22568499,
        250.22568499,  255.61968499,  259.66668499,  263.42068499,
        267.61668499,  272.30668499,  278.29368499,  282.29668499,
        286.33668499,  290.13168499,  292.92868499,  299.50368499,
        305.37868499,  310.45668499,  314.57368499,  317.46568499,
        320.89568499,  323.61268499,  328.81068499,  333.60468

In [ ]:
event_times = np.asarray(trials['firstMovement_times'])

valid_trials = np.isfinite(event_times)

event_times = event_times[valid_trials]

print("Valid trials:", len(event_times))

Valid trials: 653


In [ ]:
#LFADS에 넣을 핵심 데이터 만들기

bin_size = 0.010      # 10 ms
t_before = 0.5        # movement 전 500 ms
t_after = 1.0         # movement 후 1000 ms

bin_edges = np.arange(
    -t_before,
    t_after + bin_size,
    bin_size
)

bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

print("Time bins:", len(bin_centers))

Time bins: 150


In [ ]:
#M1 뉴런들의 spike time만 준비

spike_times = np.asarray(spikes['times'])
spike_clusters = np.asarray(spikes['clusters'])

unit_spikes = {}

for cid in m1_cids:
    unit_spikes[cid] = spike_times[spike_clusters == cid]

print("Number of M1 units:", len(unit_spikes))

Number of M1 units: 5


In [ ]:
#Trial × Time × Neuron tensor 생성

n_trials = len(event_times)
n_bins = len(bin_centers)
n_units = len(m1_cids)

X = np.zeros(
    (n_trials, n_bins, n_units),
    dtype=np.int16
)

for trial_idx, event_t in enumerate(event_times):

    for unit_idx, cid in enumerate(m1_cids):

        st = unit_spikes[cid]

        # event 기준 상대 시간으로 변환
        rel_spikes = st - event_t

        counts, _ = np.histogram(
            rel_spikes,
            bins=bin_edges
        )

        X[trial_idx, :, unit_idx] = counts

print("LFADS input shape:", X.shape)

#결과 (trials, time bins, neurons)

LFADS input shape: (653, 150, 5)


In [ ]:
print(X[0].shape)
#첫번째 single-trial 전체

(150, 5)


In [ ]:
#movement-aligned 기준 분석
event_times = np.asarray(trials['firstMovement_times'])

In [ ]:
#stimulus-aligned 기준 분석
event_times = np.asarray(trials['stimOn_times'])

#둘 중 하나 선택임

In [ ]:
import numpy as np

# 실제 M1 good-unit ID
correct_cids = np.asarray(clusters["cluster_id"])[m1_good_mask]

# 앞서 사용한 ID와 같은지 확인
if not np.array_equal(np.asarray(m1_cids), correct_cids):
    raise ValueError(
        "사용한 뉴런 ID가 실제 cluster ID와 다릅니다.\n"
        "m1_cids = np.asarray(clusters['cluster_id'])[m1_good_mask]\n"
        "로 수정한 뒤, unit_spikes와 X를 생성하는 부분을 다시 실행하세요."
    )

# LFADS 입력 데이터 확인
X = np.asarray(X)

if X.ndim != 3 or 0 in X.shape:
    raise ValueError(
        f"비어 있지 않은 (trials, time bins, neurons) 배열이 필요합니다: {X.shape}"
    )

if not np.isfinite(X).all():
    raise ValueError("X에 NaN 또는 무한대가 있습니다.")

if np.any(X < 0) or np.any(X != np.floor(X)):
    raise ValueError("X는 음수가 없는 정수 spike count여야 합니다.")

if X.shape[2] != len(correct_cids):
    raise ValueError("X의 뉴런 수와 선택한 cluster ID 수가 다릅니다.")

unit_totals = X.sum(axis=(0, 1))


print("X.shape:", X.shape)
print("Total spike counts:", int(X.sum()))
print("Units with zero spikes in all windows:", int(np.sum(unit_totals == 0)))
print("Cluster ID check: PASS")

X.shape: (653, 150, 5)
Total spike counts: 29515
Units with zero spikes in all windows: 0
Cluster ID check: PASS


In [ ]:
#터미널에서
python /home/changhyun/LFADS_pra/02_check_env.py

In [ ]:
%pip install h5py
#%pip: 현재 Notebook 커널의 Python 환경에 설치

import h5py

print("h5py version:", h5py.__version__)
#설치확인

Note: you may need to restart the kernel to use updated packages.
h5py version: 3.16.0


In [ ]:
print("Cluster IDs match:", np.array_equal(m1_cids, correct_cids))

Cluster IDs match: True


In [ ]:
from pathlib import Path
import h5py

valid_mask = np.asarray(valid_trials, dtype=bool)
raw_events = np.asarray(event_times, dtype=float)

assert len(raw_events) == len(valid_mask)
assert np.sum(valid_mask) == X.shape[0]

events = raw_events[valid_mask]

print("Filtered events:", len(events))
print("X trials:", X.shape[0])

edges = np.asarray(bin_edges, dtype=float)

# 원래 IBL trials 배열에서의 trial 번호를 보존
trial_ids = np.flatnonzero(np.asarray(valid_trials, dtype=bool))

n_trials, n_bins, n_units = X.shape

if n_trials < 2:
    raise ValueError("학습/검증 분리를 위해 최소 두 개의 trial이 필요합니다.")

if events.shape != (n_trials,) or len(trial_ids) != n_trials:
    raise ValueError("X, event_times, valid_trials의 trial 수가 맞지 않습니다.")

if not np.isfinite(events).all():
    raise ValueError("event_times에 유효하지 않은 시간이 있습니다.")

if (
    len(edges) != n_bins + 1
    or not np.allclose(np.diff(edges), bin_size)
):
    raise ValueError("X의 time bin 수 또는 bin_size가 bin_edges와 맞지 않습니다.")

# 전체 기록 시간 기준으로 정렬 후 분리
order = np.argsort(events)
cut = min(n_trials - 1, max(1, int(n_trials * 0.8)))

train_idx = order[:cut]
valid_idx = order[cut:]

# Train과 validation에 동일한 기록 구간이 들어가지 않도록 처리
first_valid_start = np.min(events[valid_idx] + edges[0])
train_ends = events[train_idx] + edges[-1]

keep = train_ends < first_valid_start
n_excluded = int(np.sum(~keep))
train_idx = train_idx[keep]

if len(train_idx) == 0:
    raise ValueError("Window 겹침을 제외하니 학습 trial이 없습니다.")

# 저장 위치: 현재 프로젝트 폴더 아래 lfads_data
out_dir = Path.cwd() / "lfads_data"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"m1_{pid}.h5"

# 'x': 같은 파일이 이미 있으면 덮어쓰지 않고 중단
with h5py.File(out_path, "x") as h5:

    for split, idx in (("train", train_idx), ("valid", valid_idx)):
        counts = X[idx]

        h5.create_dataset(
            f"{split}_encod_data",
            data=counts,
            compression="gzip",
        )
        h5.create_dataset(
            f"{split}_recon_data",
            data=counts,
            compression="gzip",
        )

        # 나중에 결과를 원래 trial 및 행동 데이터와 연결하기 위한 정보
        h5.create_dataset(f"{split}_trial_ids", data=trial_ids[idx])
        h5.create_dataset(f"{split}_event_times_s", data=events[idx])

    h5.create_dataset("cluster_ids", data=correct_cids)
    h5.create_dataset("bin_edges_s", data=edges)

    h5.attrs["bin_size_s"] = float(bin_size)
    h5.attrs["pid"] = str(pid)
    h5.attrs["eid"] = str(eid)
    h5.attrs["split_method"] = "chronological_80_20_with_overlap_purge"

print("Train shape:", X[train_idx].shape)
print("Valid shape:", X[valid_idx].shape)
print("Excluded overlapping train windows:", n_excluded)
print("Saved:", out_path.resolve())

Filtered events: 653
X trials: 653
Train shape: (522, 150, 5)
Valid shape: (131, 150, 5)
Excluded overlapping train windows: 0
Saved: /home/changhyun/LFADS_pra/lfads_data/m1_0ffa052a-fe3f-46c2-999d-43bd56b7826e.h5


In [ ]:
#lfads-torch 설치

deactivate
conda activate lfads-torch
cd /home/changhyun/LFADS_pra
git clone https://github.com/arsedler9/lfads-torch.git
python -m pip install -e ./lfads-torch

In [ ]:
python -c "import lfads_torch; print('LFADS import OK')"
python -c "import torch; print('PyTorch:', torch.__version__)"

#LFADS import OK
#PyTorch: 1.13.1+cu117
#이렇게 뜨면 LFADS 설치 완료

SyntaxError: invalid syntax (2505839989.py, line 1)

In [ ]:
python -c "import h5py; f=h5py.File('lfads_data/m1_0ffa052a-fe3f-46c2-999d-43bd56b7826e.h5','r'); print('Train:', f['train_encod_data'].shape); print('Valid:', f['valid_encod_data'].shape); f.close()"
#HDF5 데이터 확인

##모델학습##

In [ ]:
cd /home/changhyun/LFADS_pra/lfads-torch

cp configs/model/nlb_mc_maze.yaml configs/model/ibl_m1.yaml #configs/model/nlb_mc_maze.yaml를 복사해서 configs/model/ibl_m1.yaml라는 파일로 분석에 사용하겠다.

#실행 후 configs/model/ibl_m1.yaml을 열고 수정

In [ ]:
hostname
echo "$SLURM_JOB_ID"
nvidia-smi

#GPU 할당받은 node에서 실행

In [ ]:
#정상적으로 할당 됐다면 LFADS 환경 활성화
conda activate lfads-torch

python -c "import torch; print(torch.cuda.is_available())"
#TURE가 나오면 LFADS 환경에서 GPU 사용 가능

cd /home/changhyun/LFADS_pra/lfads-torch
python scripts/run_ibl_m1.py
#이후 실행

##Latent Factors 추출##

In [ ]:
cd /home/changhyun/LFADS_pra/lfads-torch
find runs -type f | tail -n 20
#학습 결과 확인

In [ ]:
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# LFADS 결과 파일
run_dir = Path(
    "/home/changhyun/LFADS_pra/lfads-torch/runs/"
    "lfads-torch-example/ibl_m1/261012_162141_ibl_m1"
)

h5_files = list(run_dir.glob("lfads_output_*.h5"))
if len(h5_files) != 1:
    raise RuntimeError(f"LFADS 출력 파일을 확인하세요: {h5_files}")

with h5py.File(h5_files[0], "r") as f:
    print("Available keys:", list(f.keys()))
    factors = f["valid_factors"][:]
    edges = f["bin_edges_s"][:]

print("Factors shape:", factors.shape)

# Trial × Time × Factors
n_trials, n_bins, n_factors = factors.shape

if n_factors < 3:
    raise ValueError("3D PCA에는 최소 3개의 factor가 필요합니다.")

# 모든 Validation trial과 시간점으로 PCA 학습
pca = PCA(n_components=3)

Z = pca.fit_transform(
    factors.reshape(-1, n_factors)
).reshape(n_trials, n_bins, 3)

print("Explained variance ratio:", pca.explained_variance_ratio_)

# Trial 평균 trajectory
mean_trajectory = Z.mean(axis=0)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

# 개별 Trial 5개
for i in range(min(5, n_trials)):
    ax.plot(
        Z[i, :, 0],
        Z[i, :, 1],
        Z[i, :, 2],
        alpha=0.35,
        linewidth=1,
    )

# Trial 평균 trajectory
ax.plot(
    mean_trajectory[:, 0],
    mean_trajectory[:, 1],
    mean_trajectory[:, 2],
    color="black",
    linewidth=3,
    label="Mean trajectory",
)

# First movement onset (t=0)
time = (edges[:-1] + edges[1:]) / 2
movement_idx = np.argmin(np.abs(time))

ax.scatter(
    *mean_trajectory[movement_idx],
    color="red",
    s=70,
    label="Movement onset",
)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title("IBL M1 LFADS Latent Trajectory")
ax.legend()

plt.tight_layout()

out_path = run_dir / "m1_lfads_pca_3d.png"
plt.savefig(out_path, dpi=200)
plt.show()

print("Saved:", out_path)

#/home/changhyun/LFADS_pra/lfads-torch/scripts/plot_m1_factors.py 들어가서 이걸로 코드를 변경

In [ ]:
python scripts/plot_m1_factors.py